# Notebook 3: Interventions and Qualitative Analysis

Colab-ready workflow for running held-out residual-patch interventions on candidate circuits and interpreting the results.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/jacobposchl/model-interpretability.git'
REPO_DIR = Path('/content/model_interpretability' if IN_COLAB else Path.cwd()).resolve()

if IN_COLAB:
    if REPO_DIR.exists():
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'], check=True)

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/flow_circuits')
    DATA_ROOT  = Path('/content/drive/MyDrive/ctls/data')
else:
    if not (REPO_DIR / 'flow_circuits').exists():
        raise RuntimeError('Run this notebook from the repository root or in Google Colab.')
    os.chdir(REPO_DIR)
    DRIVE_ROOT = REPO_DIR / 'artifacts' / 'local_notebook_runtime'
    DATA_ROOT  = DRIVE_ROOT / 'data'

BACKBONE_ROOT   = DRIVE_ROOT / 'backbones'
EXPERIMENT_ROOT = DRIVE_ROOT / 'experiments'
NOTEBOOK_ROOT   = DRIVE_ROOT / 'notebook_runs'
for path in (DRIVE_ROOT, DATA_ROOT, BACKBONE_ROOT, EXPERIMENT_ROOT, NOTEBOOK_ROOT):
    path.mkdir(parents=True, exist_ok=True)

CONFIG_ROOT = REPO_DIR / 'configs' / 'flow'
print(f"Repo      : {REPO_DIR}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"Data      : {DATA_ROOT}")

In [ ]:
from flow_circuits.data import build_cifar10_splits
from flow_circuits.interventions import run_circuit_interventions
from flow_circuits.training import collect_model_outputs, load_components_from_checkpoint

## Step 1 — Configure Your Run

Edit the cell below. Everything else runs automatically.

---

**`TRAINING_MODE`** — controls how long training runs:

| Mode | Epochs | Time on T4 GPU | Use for |
|------|--------|----------------|---------|
| `'smoke'` | 1 per phase | ~2 min | Verifying the pipeline runs end-to-end |
| `'debug'` | 5 per phase | ~30 min | Checking loss curves move in the right direction |
| `'full'` | Config defaults (20+20) | 3–6 hr | Scientifically valid results |

> **Note:** Results from `'smoke'` or `'debug'` are **not** scientifically meaningful — the model has not converged.

---

**`CONFIG_NAME`** — selects the model configuration:

- `'resnet18_base'` — Phases A + B only. No trajectory alignment (Phase C skipped). Faster.
- `'resnet18_aligned'` — Full pipeline with Phase C trajectory alignment sweep.

---

**Path overrides** — set these to reuse files you've already computed. Leave as `None` to auto-generate.

In [ ]:
# ── Training mode ─────────────────────────────────────────────────────────────
#   'smoke' = 1 epoch, ~2 min    (pipeline check only — results not meaningful)
#   'debug' = 5 epochs/phase, ~30 min on GPU  (trends are directionally valid)
#   'full'  = config defaults, 3-6 hr on T4   (scientifically valid results)
TRAINING_MODE = 'smoke'  # <- change me

# ── Model configuration ───────────────────────────────────────────────────────
#   'resnet18_base'    = Phases A+B only (no trajectory alignment)
#   'resnet18_aligned' = Full pipeline with Phase C trajectory alignment sweep
CONFIG_NAME = 'resnet18_aligned'  # <- change me

# ── Reuse existing artifacts (set to None to auto-generate) ───────────────────
BACKBONE_WEIGHTS_PATH = None  # path to a trained ResNet18-CIFAR10 .pt file
CHECKPOINT_PATH       = None  # path to a trained flow-circuits final.pt file
CIRCUITS_PATH         = None  # path to a candidate_circuits.json file

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = str(NOTEBOOK_ROOT / 'nb03')

In [ ]:
import copy
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml

# ── Training mode settings ────────────────────────────────────────────────────
_MODE_SETTINGS = {
    'smoke': {
        'label': 'Smoke test (~2 min) — pipeline check only, results NOT meaningful',
        'batch_size': 32, 'num_workers': 0,
        'phase_epochs': {'phase_a': 1, 'phase_b': 1, 'phase_c': 1},
        'baseline_fit': 128, 'baseline_eval': 128, 'val_images': 64,
        'align_pairs': 256, 'disc_images': 256, 'disc_bootstrap': 4, 'interv_images': 128,
    },
    'debug': {
        'label': 'Debug run (~30 min on GPU) — trends meaningful, not publication quality',
        'batch_size': 64, 'num_workers': 2,
        'phase_epochs': {'phase_a': 5, 'phase_b': 5, 'phase_c': 5},
        'baseline_fit': 512, 'baseline_eval': 512, 'val_images': 512,
        'align_pairs': 1024, 'disc_images': 2000, 'disc_bootstrap': 10, 'interv_images': 1000,
    },
    'full': {
        'label': 'Full training (3-6 hr on T4) — scientifically valid results',
        'batch_size': None, 'num_workers': None, 'phase_epochs': None,
        'baseline_fit': None, 'baseline_eval': None, 'val_images': None,
        'align_pairs': None, 'disc_images': None, 'disc_bootstrap': None, 'interv_images': None,
    },
}
_ms = _MODE_SETTINGS[TRAINING_MODE]
print(f"Training mode : {TRAINING_MODE}")
print(f"  {_ms['label']}")

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLI_MODULES = {
    'flow-train': 'flow_circuits.cli.train',
    'flow-discover': 'flow_circuits.cli.discover',
    'flow-intervene': 'flow_circuits.cli.intervene',
}

def run_flow_cli(command: str, *args: str) -> None:
    def _stream(cmd):
        import os
        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"
        process = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, cwd=REPO_DIR, env=env,
        )
        while True:
            line = process.stdout.readline()
            if not line:
                break
            print(line, end="", flush=True)
        process.wait()
        if process.returncode != 0:
            raise subprocess.CalledProcessError(process.returncode, cmd)
    try:
        _stream([command, *args])
    except FileNotFoundError:
        _stream([sys.executable, '-m', CLI_MODULES[command], *args])

with open(str(CONFIG_ROOT / f'{CONFIG_NAME}.yaml'), 'r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

if not BACKBONE_WEIGHTS_PATH:
    BACKBONE_WEIGHTS_PATH = str(BACKBONE_ROOT / f"{config['backbone']['arch']}_cifar10_supervised.pt")
config['data']['data_dir'] = str(DATA_ROOT)
config['data']['download'] = False
config['backbone']['weights_path'] = str(BACKBONE_WEIGHTS_PATH)
config['logging']['checkpoint_dir'] = str(EXPERIMENT_ROOT / config['experiment']['name'])

effective_config = copy.deepcopy(config)
effective_phase_epochs = None if _ms['phase_epochs'] is None else copy.deepcopy(_ms['phase_epochs'])
if effective_phase_epochs is not None and effective_config['experiment'].get('mode', 'base') != 'aligned':
    effective_phase_epochs['phase_c'] = 0
if TRAINING_MODE != 'full':
    effective_config['data']['batch_size'] = _ms['batch_size']
    effective_config['data']['num_workers'] = _ms['num_workers']
    effective_config['training']['phase_epochs'] = effective_phase_epochs
    effective_config['training']['baseline_fit_images'] = _ms['baseline_fit']
    effective_config['training']['baseline_eval_images'] = _ms['baseline_eval']
    effective_config['training']['validation_images'] = _ms['val_images']
    effective_config['training']['alignment_max_pairs'] = _ms['align_pairs']
    effective_config['discovery']['max_images'] = _ms['disc_images']
    effective_config['discovery']['bootstrap_iterations'] = _ms['disc_bootstrap']
    effective_config['interventions']['max_images'] = _ms['interv_images']
    effective_config['logging']['checkpoint_dir'] = str(OUTPUT_DIR / f'{TRAINING_MODE}_checkpoints')

EFFECTIVE_CONFIG = OUTPUT_DIR / f'{TRAINING_MODE}_config.yaml'
EFFECTIVE_CONFIG.write_text(yaml.safe_dump(effective_config, sort_keys=False), encoding='utf-8')

PHASE_B_CHECKPOINT = Path(effective_config['logging']['checkpoint_dir']) / 'phase_b.pt'
EFFECTIVE_CHECKPOINT = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else Path(effective_config['logging']['checkpoint_dir']) / 'final.pt'
EFFECTIVE_CIRCUITS = Path(CIRCUITS_PATH) if CIRCUITS_PATH else Path(effective_config['logging']['checkpoint_dir']) / 'candidate_circuits.json'
SUMMARY_JSON = OUTPUT_DIR / 'intervention_summary.json'
print(f"Circuits    : {EFFECTIVE_CIRCUITS}")
print(f"Summary     : {SUMMARY_JSON}")
print(f"\nConfig      : {EFFECTIVE_CONFIG}")
print(f"Phases      : {effective_config['training']['phase_epochs']}")
print(f"Phase B ckpt: {PHASE_B_CHECKPOINT}")
print(f"Checkpoint  : {EFFECTIVE_CHECKPOINT}")

## Step 2 — Train, Discover, and Run Interventions

The cell below runs the full pipeline:
1. Trains the encoder (skipped if checkpoint exists)
2. Discovers candidate circuits (skipped if artifact exists)
3. Runs residual-patch interventions on the held-out test set

**What the intervention test does:**
For each candidate circuit, we zero out its active nodes' residual activations and
measure the drop in the model's classification confidence. A genuine circuit should
cause a *larger* confidence drop for images that activate it (members) than for
matched control images (non-members), random nodes, or random spatial cells.

In [ ]:
RESUME_CHECKPOINT = PHASE_B_CHECKPOINT if (not EFFECTIVE_CHECKPOINT.exists() and PHASE_B_CHECKPOINT.exists()) else None
if not EFFECTIVE_CHECKPOINT.exists():
    if RESUME_CHECKPOINT is not None:
        print("Final checkpoint not found — resuming from Phase B checkpoint...")
        print(f"  {RESUME_CHECKPOINT}")
        run_flow_cli('flow-train', '--config', str(EFFECTIVE_CONFIG), '--resume', str(RESUME_CHECKPOINT))
    else:
        print("Flow model checkpoint not found — starting training...")
        run_flow_cli('flow-train', '--config', str(EFFECTIVE_CONFIG))
else:
    print(f"Flow model checkpoint found — skipping training.")

if not EFFECTIVE_CIRCUITS.exists():
    print("\nCircuit artifact not found — running discovery...")
    run_flow_cli('flow-discover', '--checkpoint', str(EFFECTIVE_CHECKPOINT), '--output', str(EFFECTIVE_CIRCUITS))
else:
    print(f"\nCircuit artifact found — skipping discovery.")

if not SUMMARY_JSON.exists():
    print("\nIntervention summary not found — running interventions...")
    run_flow_cli('flow-intervene', '--checkpoint', str(EFFECTIVE_CHECKPOINT), '--circuits', str(EFFECTIVE_CIRCUITS), '--output', str(SUMMARY_JSON))
else:
    print(f"\nIntervention summary found — skipping interventions.")

intervention_summary = json.loads(SUMMARY_JSON.read_text(encoding='utf-8'))
circuits_artifact = json.loads(EFFECTIVE_CIRCUITS.read_text(encoding='utf-8'))

n_validated = sum(1 for r in intervention_summary if r.get('validated'))
print(f"\n{len(intervention_summary)} circuit(s) tested, {n_validated} validated.")

## Step 3 — Interpret Intervention Results

A circuit is **validated** when ablating its active nodes causes a statistically
significant larger confidence drop for its member images than for:
- matched non-member images (same predicted class, similar confidence margin)
- random nodes at the same layers
- random spatial cells at the same layers

All three comparisons must pass Holm-corrected significance tests (p < α).

In [ ]:
import numpy as np

if not intervention_summary:
    print("No intervention results available.")
else:
    n_validated = sum(1 for r in intervention_summary if r.get('validated'))
    print(f"Results: {len(intervention_summary)} circuits tested, {n_validated} validated\n")
    print(f"  {'Circuit':>7}  {'Members':>7}  {'Controls':>8}  {'Member drop':>11}  {'Control drop':>12}  {'Validated'}")
    print(f"  {'-'*7}  {'-'*7}  {'-'*8}  {'-'*11}  {'-'*12}  {'-'*9}")
    for r in intervention_summary:
        status = "YES" if r.get('validated') else "no"
        print(
            f"  {r['circuit_id']:>7}  {r['n_members']:>7}  {r['n_controls']:>8}"
            f"  {r['mean_member_delta_margin']:>+11.4f}  {r['mean_nonmember_delta_margin']:>+12.4f}  {status}"
        )
    print()
    print("Member drop  = how much classification confidence falls when the circuit's nodes are ablated.")
    print("Control drop = same ablation applied to matched non-member images.")
    print("A validated circuit shows: member drop > control drop, > random nodes, > random cells.")
    if TRAINING_MODE == 'smoke':
        print("\n*** SMOKE MODE: results are from 1-epoch training and are not meaningful. ***")

In [ ]:
if intervention_summary:
    import numpy as np
    labels = [f"C{row['circuit_id']}" for row in intervention_summary]
    member = [row['mean_member_delta_margin'] for row in intervention_summary]
    nonmember = [row['mean_nonmember_delta_margin'] for row in intervention_summary]
    random_node = [row['mean_random_node_delta_margin'] for row in intervention_summary]
    random_cell = [row['mean_random_cell_delta_margin'] for row in intervention_summary]

    x = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(max(6, 1.2 * len(labels)), 4))
    ax.plot(x, member, marker='o', label='members (circuit images)')
    ax.plot(x, nonmember, marker='o', label='matched non-members')
    ax.plot(x, random_node, marker='o', label='random-node control')
    ax.plot(x, random_cell, marker='o', label='random-cell control')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylabel('Mean confidence drop (margin delta)')
    ax.set_title('Intervention effect: members vs controls\n(higher = larger drop when circuit is ablated)')
    ax.legend()
    plt.tight_layout()
else:
    print('No intervention rows available to visualize.')

In [ ]:
max_plots = min(4, len(circuits_artifact['circuits']))
grid_size = circuits_artifact['metadata']['grid_size']
if max_plots == 0:
    print('No candidate circuits available in the artifact.')
else:
    fig, axes = plt.subplots(1, max_plots, figsize=(4 * max_plots, 3.5))
    if max_plots == 1:
        axes = [axes]
    for axis, circuit in zip(axes, circuits_artifact['circuits'][:max_plots]):
        import numpy as np
        heatmap = np.zeros((grid_size, grid_size), dtype=float)
        for _, cell_idx in circuit['active_nodes']:
            row = cell_idx // grid_size
            col = cell_idx % grid_size
            heatmap[row, col] += 1.0
        image = axis.imshow(heatmap, cmap='viridis')
        axis.set_title(f"Circuit {circuit['id']} footprint")
        plt.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    plt.tight_layout()

In [ ]:
validated = [row for row in intervention_summary if row.get('validated')]
if validated:
    print(f"{len(validated)} validated circuit(s):")
    for r in validated:
        print(f"  Circuit {r['circuit_id']}: {r['n_members']} members, member drop={r['mean_member_delta_margin']:+.4f}")
        print(f"    p (vs non-member): {r['corrected_p_member_vs_nonmember']:.4f}")
        print(f"    p (vs rand node) : {r['corrected_p_member_vs_random_node']:.4f}")
        print(f"    p (vs rand cell) : {r['corrected_p_member_vs_random_cell']:.4f}")
else:
    print('No circuits passed the current intervention validation rule.')
    print('This is expected with smoke/debug mode training or when no circuits were discovered.')